In [3]:
import pandas as pd

ratings = pd.read_csv('../data/raw/ratings.csv')
movies = pd.read_csv('../data/raw/movies.csv')

In [4]:
user_item = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [5]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50)
matrix = svd.fit_transform(user_item)

In [6]:
import numpy as np

reconstructed = np.dot(matrix, svd.components_)
reconstructed_df = pd.DataFrame(
    reconstructed,
    index=user_item.index,
    columns=user_item.columns
)

In [7]:
def recommend_svd(user_id, top_n=5):
    user_ratings = reconstructed_df.loc[user_id]
    
    already_watched = user_item.loc[user_id]
    already_watched = already_watched[already_watched > 0].index
    
    recommendations = user_ratings.drop(already_watched)
    recommendations = recommendations.sort_values(ascending=False).head(top_n)
    
    return movies[movies['movieId'].isin(recommendations.index)][['title']]

In [8]:
recommend_svd(1)

,title
659,"Godfather, The (1972)"
793,Die Hard (1988)
922,"Godfather: Part II, The (1974)"
1067,Jaws (1975)
1445,"Breakfast Club, The (1985)"
